# 리포트 파이프라인 평가: Ver1 vs Ver2

| 버전 | 요약 | 체크리스트 | 평가 순서 |
|------|------|-----------|----------|
| **Ver1** | 골드셋 요약 컬럼 **고정** | 골드셋 문항 **고정** | ① T/F 비교 → ② 질문/모범답안 → ③ 리포트 |
| **Ver2** | `invoke()` 내부 생성 | `invoke()` 자체 생성 | ⓪ 요약 비교 → ① 체크리스트 내용 비교 → ② 질문/모범답안 → ③ 리포트 |

> 양 버전 모두 원천 인풋은 **골드셋의 resume, company_info, job_description 동일 사용**.  
> **Ver1 = 모듈 격리 평가**: 상류(요약·체크리스트)를 골드로 고정 → T/F 판정·질문·리포트 모듈 성능만 측정.  
> **Ver2 = end-to-end 평가**: 전 구간 자체 생성 → 실서비스 동작 그대로 측정.  
> Ver1과 Ver2의 점수 차이 = 상류(요약·체크리스트 생성) 단에서 새는 성능.  
> 각 평가 단에서 지표 + **차이 큰 예시 / 유사한 예시** 1쌍씩 출력.

In [19]:
# [설정] 경로/모듈 로드 + 수정 골드셋 로드. report2의 생성 함수들과 OpenAI 클라이언트 준비.
from pathlib import Path
import sys, json, time, textwrap
from difflib import SequenceMatcher
from typing import List

import pandas as pd
from pydantic import BaseModel, Field

PROJECT_ROOT = Path('C:/project_skn/final/Final_project')
BACKEND_DIR  = PROJECT_ROOT / 'backend'
EVAL_DIR     = BACKEND_DIR / 'common' / 'eval'
sys.path.insert(0, str(BACKEND_DIR))

from common.report2 import (
    invoke, sum_resume, sum_company, sum_jd,
    check_resume_fit, make_report, make_interview_questions,
    _get_openai_client,
)

client      = _get_openai_client()
JUDGE_MODEL = 'gpt-4o-mini'

goldset_df = pd.read_csv(EVAL_DIR / 'goldset_mock_data_fixed.csv', encoding='utf-8-sig')
goldset_df.columns = goldset_df.columns.str.strip()
print(f'골드셋 로드 완료: {goldset_df.shape}')
print(goldset_df.columns.tolist())

def parse_json_cell(v):
    if isinstance(v, (dict, list)):
        return v
    return json.loads(v)


골드셋 로드 완료: (20, 11)
['set_id', 'company_info', 'job_description', 'resume', 'checklist', 'report', 'questions_answers', 'quality_label', 'company_summary', 'jd_summary', 'resume_summary']


In [6]:
# [judge 정의] 질문/리포트/체크리스트내용/요약 비교용 LLM judge 스키마와 프롬프트.
# 모든 점수는 1~5 정수, PASS 기준은 각 프롬프트에 명시.
# ── Pydantic 스키마 ──────────────────────────────────────────────────────

class QuestionJudgeResult(BaseModel):
    set_id: int
    question_coverage_score:    int   = Field(description='골드 질문 평가 의도 포함 정도. 1~5')
    answer_similarity_score:    int   = Field(description='모범답안 의미 일치도. 1~5')
    overall_question_score:     int   = Field(description='전체 품질. 1~5')
    pass_fail:                  str   = Field(description='PASS 또는 FAIL')
    matched_gold_question_count: int  = Field(description='골드 10개 중 의미상 대응 수')
    judge_comment:              str

class ReportJudgeResult(BaseModel):
    set_id: int
    grade_match:                 bool  = Field(description='overall_grade 일치 여부')
    checklist_result_match_rate: float = Field(description='result 일치율 0~1')
    summary_similarity_score:    int   = Field(description='요약 의미 일치도. 1~5')
    analysis_similarity_score:   int   = Field(description='역량/적합도 분석 일치도. 1~5')
    overall_report_score:        int   = Field(description='최종 리포트 품질. 1~5')
    pass_fail:                   str   = Field(description='PASS 또는 FAIL')
    judge_comment:               str

# ── 판정 프롬프트 ─────────────────────────────────────────────────────────

Q_SYS = (
    '너는 면접 질문/모범답안을 비교하는 LLM judge다. '
    '질문 의도(필수기술, 회사연결, 협업, 장애대응, 입사계획 등)이 같으면 대응된 것으로 본다. '
    '표현과 순서 차이는 감점 안 한다. 근거 없이 일반론으로 흐를 때만 감점한다. '
    'PASS: overall_question_score≥4 이고 matched_gold_question_count≧7. 점수는 1~5 정수.'
)

R_SYS = (
    '너는 채용 평가 리포트를 검수하는 LLM judge다. '
    'result T/F 판단 차이와 overall_grade 일치를 가장 중요하게 본다. '
    '근거 없는 환각/추가는 감점, 입력 근거 있는 합리적 서술은 감점 안 한다. '
    'PASS: overall_report_score≥4 이고 중대한 환각 없을 때. 점수는 1~5 정수.'
)

def _judge(system, payload, schema):
    resp = client.beta.chat.completions.parse(
        model=JUDGE_MODEL,
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user',   'content': json.dumps(payload, ensure_ascii=False)},
        ],
        response_format=schema,
    )
    return resp.choices[0].message.parsed.model_dump()

def judge_questions(sid, gold_qs, gen_qs):
    return _judge(Q_SYS, {'set_id': int(sid), 'gold': gold_qs, 'generated': gen_qs}, QuestionJudgeResult)

def judge_report(sid, gold_rpt, gen_rpt):
    gold_cl = gold_rpt.get('checklist', [])
    gen_cl  = gen_rpt.get('checklist', []) if isinstance(gen_rpt, dict) else []
    n = len(gold_cl)
    rate = sum(
        1 for i, g in enumerate(gold_cl)
        if i < len(gen_cl) and g['result'] == gen_cl[i].get('result')
    ) / n if n else 0.0
    return _judge(R_SYS,
                  {'set_id': int(sid), 'checklist_result_match_rate': rate,
                   'gold': gold_rpt, 'generated': gen_rpt},
                  ReportJudgeResult)

# ── 체크리스트 내용 judge (Ver2용) ───────────────────────────────────────

class ChecklistContentJudgeResult(BaseModel):
    set_id: int
    content_similarity_score: int = Field(description='문항 내용 의미 유사도. 1~5')
    coverage_score:           int = Field(description='골드 핵심 평가 기준 포함 정도. 1~5')
    extra_noise_score:        int = Field(description='근거 약하거나 불필요한 항목이 적은 정도. 1~5')
    overall_score:            int = Field(description='전체 품질 및 골드 정렬도. 1~5')
    pass_fail:                str = Field(description='PASS 또는 FAIL')
    matched_gold_count:       int = Field(description='골드 10개 중 의미상 대응된 항목 수')
    missing_gold_criteria:    List[str] = Field(description='생성 체크리스트에서 누락된 골드 기준')
    judge_comment:            str

CL_SYS = (
    '너는 채용 평가 체크리스트를 비교하는 LLM judge다. '
    '골드 체크리스트와 생성 체크리스트의 "문항 내용"을 비교해 골드 기준을 얼마나 재현했는지 평가한다. '
    '표현, 어순, 항목 순서, 길이 차이는 감점하지 않는다. 평가 의도가 같으면 대응으로 본다. '
    '골드 평가 영역(학력, 경력, 필수기술, 우대기술, 도메인, 개발 전반, 장애/품질, 비개발 협업, 보안, 지원동기)이 '
    '다뤄지면 매칭으로 인정한다. 입력 근거와 무관하거나 너무 일반적인 항목만 감점한다. '
    'PASS: overall_score≥4 이고 matched_gold_count≥7. 점수는 1~5 정수.'
)

def judge_checklist_content(sid, gold_cl, gen_cl):
    return _judge(CL_SYS,
                  {'set_id': int(sid),
                   'gold_checklist': [c['content'] for c in gold_cl],
                   'generated_checklist': [c.get('content') for c in gen_cl]},
                  ChecklistContentJudgeResult)


# ── 요약 judge (Ver2 ⓪단용): 생성 요약 vs 골드 요약 ──────────────────────

class SummaryJudgeResult(BaseModel):
    set_id: int
    resume_summary_score:  int = Field(description='이력서 요약의 핵심 정보 보존/의미 일치도. 1~5')
    company_summary_score: int = Field(description='회사 요약의 핵심 정보 보존/의미 일치도. 1~5')
    jd_summary_score:      int = Field(description='JD 요약의 핵심 정보 보존/의미 일치도. 1~5')
    missing_info:          List[str] = Field(description='생성 요약에서 누락된 중요 정보(경력 연수, 기술명 등)')
    hallucination:         List[str] = Field(description='골드/원본에 없는데 생성 요약에 추가된 내용')
    overall_score:         int = Field(description='세 요약 종합 품질. 1~5')
    pass_fail:             str = Field(description='PASS 또는 FAIL')
    judge_comment:         str

S_SYS = (
    '너는 채용 데이터 요약을 비교하는 LLM judge다. '
    '생성 요약(이력서/회사/JD)이 골드 요약과 같은 핵심 정보(경력 연수, 학력, 기술명, 도메인, '
    '필수/우대 기술, 협업·보안 관련 언급)를 담고 있는지 평가한다. '
    '문체와 표현 차이는 감점하지 않는다. 핵심 정보 누락과 근거 없는 추가만 감점한다. '
    'PASS: overall_score≥4 이고 체크리스트 판단에 영향을 줄 누락이 없을 때. 점수는 1~5 정수.'
)

def judge_summaries(sid, gold, gen):
    return _judge(S_SYS, {'set_id': int(sid), 'gold': gold, 'generated': gen}, SummaryJudgeResult)


In [17]:
# [헬퍼] 지표 계산 함수(T/F 분류지표, 질문/리포트 judge 집계)와
# 각 단의 차이 큰/유사한 예시를 골드 vs 생성으로 나란히 출력하는 함수들.
# ── 지표 계산 헬퍼 ────────────────────────────────────────────────────────

def tf_metrics(df):
    valid = df[df['predicted'].apply(lambda x: isinstance(x, bool))].copy()
    tp = ((valid['expected']) & (valid['predicted'] == True)).sum()
    tn = ((~valid['expected']) & (valid['predicted'] == False)).sum()
    fp = ((~valid['expected']) & (valid['predicted'] == True)).sum()
    fn = ((valid['expected']) & (valid['predicted'] == False)).sum()
    p = tp / (tp + fp) if tp + fp else 0
    r = tp / (tp + fn) if tp + fn else 0
    return pd.DataFrame([{
        'count': len(valid), 'accuracy': valid['is_correct'].mean(),
        'precision': p, 'recall': r, 'f1': 2*p*r/(p+r) if p+r else 0,
        'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn,
    }])

def criterion_acc(df):
    valid = df[df['predicted'].apply(lambda x: isinstance(x, bool))]
    return (valid.groupby('criterion_id')
            .agg(accuracy=('is_correct', 'mean'),
                 gold_true=('expected', 'mean'),
                 pred_true=('predicted', 'mean'))
            .reset_index())

def q_metrics(df):
    return pd.DataFrame([{
        'count':        len(df),
        'pass_rate':    (df['pass_fail'] == 'PASS').mean(),
        'avg_score':    df['overall_question_score'].mean(),
        'avg_matched':  df['matched_gold_question_count'].mean(),
        'avg_coverage': df['question_coverage_score'].mean(),
        'avg_ans_sim':  df['answer_similarity_score'].mean(),
    }])

def r_metrics(df):
    return pd.DataFrame([{
        'count':            len(df),
        'grade_match_rate': df['grade_match'].mean(),
        'pass_rate':        (df['pass_fail'] == 'PASS').mean(),
        'avg_score':        df['overall_report_score'].mean(),
        'avg_cl_match':     df['checklist_result_match_rate'].mean(),
        'avg_analysis_sim': df['analysis_similarity_score'].mean(),
        'avg_summary_sim':  df['summary_similarity_score'].mean(),
    }])

# ── 예시 출력 헬퍼 ────────────────────────────────────────────────────────

SEP = '=' * 72

def show_tf_v1(df):
    """Ver1: 골드 체크리스트 고정 → 오답/정답 예시"""
    wrong = df[~df['is_correct']]
    right = df[df['is_correct']]
    print(SEP)
    print('【 오답 예시 — T/F 불일치 】')
    if len(wrong):
        w = wrong.iloc[0]
        print(f'  set_id={w["set_id"]}  문항{w["criterion_id"]}')
        print(f'  체크리스트 항목: {w["content"]}')
        print(f'  골드정답={w["expected"]}  →  생성={w["predicted"]}')
    print()
    print('【 정답 예시 — T/F 일치 】')
    if len(right):
        r = right.iloc[0]
        print(f'  set_id={r["set_id"]}  문항{r["criterion_id"]}')
        print(f'  체크리스트 항목: {r["content"]}')
        print(f'  골드정답={r["expected"]}  =  생성={r["predicted"]}')
    print(SEP)

def show_checklist_content_ex(judge_df, gold_cl_map, gen_cl_map):
    """Ver2: judge 점수 최저/최고 set의 골드 vs 생성 체크리스트 문항 내용을 나란히 출력"""
    worst = judge_df.loc[judge_df['overall_score'].idxmin()]
    best  = judge_df.loc[judge_df['overall_score'].idxmax()]
    print(SEP)
    for label, row in [('▼ 내용 차이 큰 예시', worst), ('▲ 내용 유사한 예시', best)]:
        sid = row['set_id']
        gold_cl = gold_cl_map.get(sid, [])
        gen_cl  = gen_cl_map.get(sid, [])
        print(f'【 {label} 】  set_id={sid}  overall={row["overall_score"]}  matched={row["matched_gold_count"]}/10')
        print(f'  judge: {str(row["judge_comment"])[:90]}')
        for i in range(max(len(gold_cl), len(gen_cl))):
            g = gold_cl[i]['content'] if i < len(gold_cl) else '-'
            n = gen_cl[i].get('content', '-') if i < len(gen_cl) else '-'
            print(f'  {i+1:>2}. [골드] {str(g)[:52]}')
            print(f'      [생성] {str(n)[:52]}')
        print()
    print(SEP)

def show_question_ex(judge_df, gold_map, gen_map):
    worst = judge_df.loc[judge_df['overall_question_score'].idxmin()]
    best  = judge_df.loc[judge_df['overall_question_score'].idxmax()]
    print(SEP)
    for label, row in [('▼ 낮은 유사도 예시', worst), ('▲ 높은 유사도 예시', best)]:
        sid = row['set_id']
        gq  = (gold_map.get(sid) or [{}])[0]
        gn  = (gen_map.get(sid) or [{}])[0]
        gn  = gn if isinstance(gn, dict) else {}
        print(f'【 {label} 】  set_id={sid}  점수={row["overall_question_score"]}')
        print(f'  [골드 질문]  {gq.get("question", "N/A")}')
        print(f'  [생성 질문]  {gn.get("question", "N/A")}')
        print(f'  [골드 답안]  {str(gq.get("answer", ""))[:100]}')
        print(f'  [생성 답안]  {str(gn.get("answer", ""))[:100]}')
        print()
    print(SEP)

def show_report_ex(judge_df, gold_map, gen_map):
    worst = judge_df.loc[judge_df['overall_report_score'].idxmin()]
    best  = judge_df.loc[judge_df['overall_report_score'].idxmax()]
    print(SEP)
    for label, row in [('▼ 낮은 유사도 예시', worst), ('▲ 높은 유사도 예시', best)]:
        sid = row['set_id']
        gr  = gold_map.get(sid, {})
        gn  = gen_map.get(sid, {})
        print(f'【 {label} 】  set_id={sid}  점수={row["overall_report_score"]}')
        print(f'  [골드] 등급={gr.get("overall_grade","?")}')
        print(f'         요약: {str(gr.get("overall_summary",""))[:90]}')
        print(f'         의견: {str(gr.get("final_comment",""))[:90]}')
        print(f'  [생성] 등급={gn.get("overall_grade","?")}')
        print(f'         요약: {str(gn.get("overall_summary",""))[:90]}')
        print(f'         의견: {str(gn.get("final_comment",""))[:90]}')
        print()
    print(SEP)


def show_summary_ex(judge_df, gold_sum_map, gen_sum_map):
    """⓪단: judge 점수 최저/최고 set의 이력서 요약을 골드 vs 생성 전문 출력"""
    def wrap(label, text):
        print(f'  [{label}]')
        for line in textwrap.wrap(str(text), width=70):
            print(f'    {line}')
    worst = judge_df.loc[judge_df['overall_score'].idxmin()]
    best  = judge_df.loc[judge_df['overall_score'].idxmax()]
    print(SEP)
    for label, row in [('▼ 차이 큰 예시', worst), ('▲ 유사한 예시', best)]:
        sid = row['set_id']
        print(f'【 {label} 】  set_id={sid}  overall={row["overall_score"]}')
        wrap('judge 코멘트', row['judge_comment'])
        print(f'  누락: {row["missing_info"]}')
        print(f'  환각: {row["hallucination"]}')
        wrap('골드 이력서요약', gold_sum_map.get(sid, {}).get('resume_summary', ''))
        wrap('생성 이력서요약', gen_sum_map.get(sid, {}).get('resume_summary', ''))
        print()
    print(SEP)


---
## 사전 준비: 파이프라인 실행

- **Ver1**: 골드 체크리스트(문항+result) 고정, T/F 재판정은 ① 평가 전용
- **Ver2**: `invoke()` 전체 파이프라인 실행 (요약~리포트 모두 내부에서 자체 생성)

> 원천 인풋은 골드셋의 `resume`, `company_info`, `job_description` 동일 사용.
> Ver1은 요약을 새로 만들지 않으므로 점수 변화가 하류 모듈(T/F 판정·질문·리포트)의 성능으로 직접 해석됨.

In [8]:
# [Ver1 생성] 골드 요약 컬럼 + 골드 체크리스트 고정.
# ① T/F 평가용: check_resume_fit으로 모델이 재판정한 결과 (골드 정답과 채점 비교)
# ②③ 질문/리포트용: 골드 체크리스트(문항+result)를 그대로 입력 → 출력만 격리 비교

v1_fit_map       = {}   # {set_id: list[{content, result}]}  ①용: 모델 재판정 T/F
v1_questions_map = {}   # {set_id: ...}  ②용: 골드 체크리스트 고정으로 생성된 질문
v1_reports_map   = {}   # {set_id: ...}  ③용: 골드 체크리스트 고정으로 생성된 리포트
v1_errors        = []

for idx, row in goldset_df.iterrows():
    sid = row['set_id']
    print(f'[Ver1] {idx+1}/{len(goldset_df)} set_id={sid}')
    try:
        gold_cl = parse_json_cell(row['checklist'])['checklist']  # [{content, result}]

        # 골드 요약 컬럼 고정 (새로 생성하지 않음)
        r_sum = str(row['resume_summary'])
        c_sum = str(row['company_summary'])
        j_sum = str(row['jd_summary'])

        # ── ① T/F 평가 전용: 골드 문항을 주고 모델이 result를 재판정 ──
        v1_fit_map[sid] = check_resume_fit(r_sum, [c['content'] for c in gold_cl])

        # ── ②③: 골드 체크리스트(문항+result)를 그대로 고정 입력 ──
        #    T/F 오류가 섞이지 않으므로 질문/리포트 모듈의 출력 품질만 측정됨
        v1_questions_map[sid] = make_interview_questions(
            resume_summary=r_sum, company_summary=c_sum,
            jd_summary=j_sum, checklist_checks=gold_cl,
        )
        v1_reports_map[sid] = make_report(
            resume_summary=r_sum, fit_checks=gold_cl,
            company_summary=c_sum, jd_summary=j_sum,
        )
    except Exception as e:
        v1_errors.append({'set_id': sid, 'error': repr(e)})
        print(f'  ERROR: {e}')
    time.sleep(0.3)

print(f'\nVer1 완료: fit={len(v1_fit_map)}, q={len(v1_questions_map)}, rpt={len(v1_reports_map)}')
if v1_errors:
    display(pd.DataFrame(v1_errors))

[Ver1] 1/20 set_id=0
[Ver1] 2/20 set_id=1
[Ver1] 3/20 set_id=2
[Ver1] 4/20 set_id=3
[Ver1] 5/20 set_id=4
[Ver1] 6/20 set_id=5
[Ver1] 7/20 set_id=6
[Ver1] 8/20 set_id=7
[Ver1] 9/20 set_id=8
[Ver1] 10/20 set_id=9
[Ver1] 11/20 set_id=10
[Ver1] 12/20 set_id=11
[Ver1] 13/20 set_id=12
[Ver1] 14/20 set_id=13
[Ver1] 15/20 set_id=14
[Ver1] 16/20 set_id=15
[Ver1] 17/20 set_id=16
[Ver1] 18/20 set_id=17
[Ver1] 19/20 set_id=18
[Ver1] 20/20 set_id=19

Ver1 완료: fit=20, q=20, rpt=20


In [9]:
# [Ver2 생성] 원본 3컬럼으로 invoke() 전체 실행 — 요약/체크리스트/질문/리포트 모두 자체 생성.
# 골드 질문/리포트 참조 맵도 여기서 준비.
# ── Ver2: 처음부터 invoke() 전체 실행 ─────────────────────────────────────

v2_invoke_map = {}   # {set_id: {questions, report}}
v2_errors     = []

for idx, row in goldset_df.iterrows():
    sid = row['set_id']
    print(f'[Ver2] {idx+1}/{len(goldset_df)} set_id={sid}')
    try:
        v2_invoke_map[sid] = invoke(
            resume_dict  = parse_json_cell(row['resume']),
            company_dict = parse_json_cell(row['company_info']),
            jd_dict      = parse_json_cell(row['job_description']),
        )
    except Exception as e:
        v2_errors.append({'set_id': sid, 'error': repr(e)})
        print(f'  ERROR: {e}')
    time.sleep(0.3)

print(f'\nVer2 완료: {len(v2_invoke_map)} / {len(goldset_df)}')
if v2_errors:
    display(pd.DataFrame(v2_errors))

# 골드 참조 데이터 (이후 셀에서 공통 사용)
gold_qa_map  = {row['set_id']: parse_json_cell(row['questions_answers'])
                for _, row in goldset_df.iterrows()}
gold_rpt_map = {row['set_id']: parse_json_cell(row['report'])
                for _, row in goldset_df.iterrows()}
print(f'골드 참조 맵 준비: QA={len(gold_qa_map)}, Report={len(gold_rpt_map)}')

[Ver2] 1/20 set_id=0
[Ver2] 2/20 set_id=1
[Ver2] 3/20 set_id=2
[Ver2] 4/20 set_id=3
[Ver2] 5/20 set_id=4
[Ver2] 6/20 set_id=5
[Ver2] 7/20 set_id=6
[Ver2] 8/20 set_id=7
[Ver2] 9/20 set_id=8
[Ver2] 10/20 set_id=9
[Ver2] 11/20 set_id=10
[Ver2] 12/20 set_id=11
[Ver2] 13/20 set_id=12
[Ver2] 14/20 set_id=13
[Ver2] 15/20 set_id=14
[Ver2] 16/20 set_id=15
[Ver2] 17/20 set_id=16
[Ver2] 18/20 set_id=17
[Ver2] 19/20 set_id=18
[Ver2] 20/20 set_id=19

Ver2 완료: 20 / 20
골드 참조 맵 준비: QA=20, Report=20


---
## ⓪ 요약 평가 (Ver2 상류 진단용)

원본 3컬럼 → `sum_resume/sum_company/sum_jd`로 생성한 요약을 골드 요약 컬럼과 LLM judge로 비교.

> Ver1(요약 고정)과 Ver2(요약 자체 생성)의 점수 차이가 클 때, 그 원인이 요약 단인지 확인하는 용도.
> 핵심 정보(경력 연수, 기술명, 학력 등) 누락/환각만 감점, 문체 차이는 무시.

In [20]:
# [⓪ 요약 평가] 생성 요약 vs 골드 요약 컬럼 비교.
# avg_overall(1~5)과 missing_info로 요약 단의 정보 유실 여부를 진단.

gold_sum_map = {row['set_id']: {'resume_summary': row['resume_summary'],
                                'company_summary': row['company_summary'],
                                'jd_summary': row['jd_summary']}
                for _, row in goldset_df.iterrows()}

gen_sum_map  = {}
s_rows, s_errors = [], []

for idx, row in goldset_df.iterrows():
    sid = row['set_id']
    print(f'[요약 평가] {idx+1}/{len(goldset_df)} set_id={sid}')
    try:
        gen = {
            'resume_summary':  sum_resume(parse_json_cell(row['resume'])),
            'company_summary': sum_company(parse_json_cell(row['company_info'])),
            'jd_summary':      sum_jd(parse_json_cell(row['job_description'])),
        }
        gen_sum_map[sid] = gen
        s_rows.append(judge_summaries(sid, gold_sum_map[sid], gen))
    except Exception as e:
        s_errors.append({'set_id': sid, 'error': repr(e)})
        print(f'  ERROR: {e}')
    time.sleep(0.2)

sum_judge_df = pd.DataFrame(s_rows)

print('\n─── ⓪ 요약 평가 지표 ───')
display(pd.DataFrame([{
    'count':       len(sum_judge_df),
    'pass_rate':   (sum_judge_df['pass_fail'] == 'PASS').mean(),
    'avg_overall': sum_judge_df['overall_score'].mean(),
    'avg_resume':  sum_judge_df['resume_summary_score'].mean(),
    'avg_company': sum_judge_df['company_summary_score'].mean(),
    'avg_jd':      sum_judge_df['jd_summary_score'].mean(),
}]))
print()
show_summary_ex(sum_judge_df, gold_sum_map, gen_sum_map)

sum_judge_df.to_csv(EVAL_DIR / 'eval_v2_summary_stage.csv', index=False, encoding='utf-8-sig')
if s_errors:
    display(pd.DataFrame(s_errors))


[요약 평가] 1/20 set_id=0
[요약 평가] 2/20 set_id=1
[요약 평가] 3/20 set_id=2
[요약 평가] 4/20 set_id=3
[요약 평가] 5/20 set_id=4
[요약 평가] 6/20 set_id=5
[요약 평가] 7/20 set_id=6
[요약 평가] 8/20 set_id=7
[요약 평가] 9/20 set_id=8
[요약 평가] 10/20 set_id=9
[요약 평가] 11/20 set_id=10
[요약 평가] 12/20 set_id=11
[요약 평가] 13/20 set_id=12
[요약 평가] 14/20 set_id=13
[요약 평가] 15/20 set_id=14
[요약 평가] 16/20 set_id=15
[요약 평가] 17/20 set_id=16
[요약 평가] 18/20 set_id=17
[요약 평가] 19/20 set_id=18
[요약 평가] 20/20 set_id=19

─── ⓪ 요약 평가 지표 ───


,count,pass_rate,avg_overall,avg_resume,avg_company,avg_jd
0,20,0.35,4.15,4.15,4.95,4.95



【 ▼ 차이 큰 예시 】  set_id=12  overall=3
  [judge 코멘트]
    생성된 요약은 일부 기술과 경력에 대한 정보가 부정확하거나 누락되었습니다. 경력 연수와 직접적으로 관련된 정보에 오류가
    있으며, 추가된 정보(TOEIC, 해커톤 수상, 기술 블로그 운영)가 확인할 수 없는 내용입니다.
  누락: ['경력 연수(총 3년 4개월의 경력이라 부정확함)', '최신 기술 관련 내용이 부족함']
  환각: ['TOEIC 700점의 영어 능력', '공공 서비스 개선 해커톤에서 우수상 수상', '기술 블로그 운영']
  [골드 이력서요약]
    남기태 님은 전남대학교 행정학과 출신으로, Python, Django, Linux, PostgreSQL, Docker 기술
    역량을 보유하고 있습니다. 지자체플랫폼에서 2년 4개월 동안 웹 개발자로 민원 데이터 수집과 관리자 통계 기능 개발. 이전
    직장에서 1년 동안 주니어 개발자로 공공 관련 기능 개선과 운영 이슈 대응. 비개발자 담당자와 요구사항을 맞추는 과정에서 쉬운
    문서화의 중요성을 배웠습니다. 보유 자격증으로는 컴퓨터활용능력 1급이 있습니다.
  [생성 이력서요약]
    남기태는 전남대학교 행정학과를 졸업한 웹 개발자로, 3년 4개월의 경력을 보유하고 있습니다. 지자체플랫폼에서 민원 데이터 수집
    및 관리자 통계 기능 개발에 참여하였고, 이전 직장에서는 공공 관련 기능 개선과 운영 이슈 대응을 담당하였습니다. 본인은
    비개발자와의 원활한 소통을 통해 문서화의 중요성을 깊이 이해하였으며, 프로젝트 일정과 품질 문제 해결에 있어 우선순위
    재조정으로 효과적인 결과를 도출한 경험이 있습니다. 기술 스택으로는 Python, Django, Linux,
    PostgreSQL, Docker를 활용하며, 컴퓨터활용능력 1급 자격증과 TOEIC 700점의 영어 능력을 보유하고
    있습니다. 그는 또한 공공 서비스 개선 해커톤에서 우수상을 수상하였고, 기술 

---
## 1. 체크리스트 평가

| | Ver1 | Ver2 |
|--|------|------|
| 체크리스트 문항 | 골드셋 문항 고정 | invoke()가 자체 생성 |
| 평가 방식 | 동일 문항이므로 **T/F 직접 대조** | 문항이 다르므로 **문항 내용을 LLM judge로 비교** |

> Ver2는 문항 자체가 다르게 생성되므로 T/F 인덱스 비교가 성립하지 않음 → 내용 비교로 평가.  
> **Ver1 예시**: 같은 항목 텍스트에서 T/F 불일치/일치 사례  
> **Ver2 예시**: judge 점수 최저/최고 set의 골드 vs 생성 문항 내용을 나란히 출력

In [21]:
# [Ver1-①] 골드 문항 고정 T/F 평가: 같은 문항에서 result만 대조 → accuracy/f1 + 오답/정답 예시.
# ── Ver1 T/F 평가 ──────────────────────────────────────────────────────────

v1_tf_rows = []
for _, row in goldset_df.iterrows():
    sid = row['set_id']
    if sid not in v1_fit_map:
        continue
    gold_cl = parse_json_cell(row['checklist'])['checklist']
    gen_cl  = v1_fit_map[sid]
    for i, (g, p) in enumerate(zip(gold_cl, gen_cl)):
        v1_tf_rows.append({
            'set_id':       sid,
            'criterion_id': i + 1,
            'content':      g['content'],  # 골드 고정이므로 항목 텍스트 동일
            'expected':     bool(g['result']),
            'predicted':    p.get('result'),
            'is_correct':   g['result'] == p.get('result'),
        })

v1_tf_df = pd.DataFrame(v1_tf_rows)

print('─── Ver1 T/F 전체 지표 ───')
display(tf_metrics(v1_tf_df))
print()
print('─── 문항(슬롯)별 정확도 ───')
display(criterion_acc(v1_tf_df))
print()
show_tf_v1(v1_tf_df)

v1_tf_df.to_csv(EVAL_DIR / 'eval_v1_tf.csv', index=False, encoding='utf-8-sig')

─── Ver1 T/F 전체 지표 ───


,count,accuracy,precision,recall,f1,TP,TN,FP,FN
0,200,0.725,0.936508,0.715152,0.810997,118,27,8,47



─── 문항(슬롯)별 정확도 ───


,criterion_id,accuracy,gold_true,pred_true
0,1,1.00,1.00,1.00
1,2,0.85,1.00,0.85
2,3,1.00,1.00,1.00
3,4,1.00,0.80,0.80
4,5,1.00,1.00,1.00
5,6,0.25,1.00,0.25
6,7,0.55,1.00,0.55
7,8,0.55,0.25,0.60
8,9,1.00,0.20,0.20
9,10,0.05,1.00,0.05



【 오답 예시 — T/F 불일치 】
  set_id=0  문항10
  체크리스트 항목: 지원 동기와 회사 서비스 사이의 연결이 명확한가?
  골드정답=True  →  생성=False

【 정답 예시 — T/F 일치 】
  set_id=0  문항1
  체크리스트 항목: 학력 무관 조건 또는 이에 준하는 실무 역량을 갖추었는가?
  골드정답=True  =  생성=True


In [15]:
# [Ver2-①] invoke 자체 생성 체크리스트의 '문항 내용'을 골드와 judge 비교 → matched/coverage + 내용 예시.
# ── Ver2 체크리스트 "내용" 비교 평가 ──────────────────────────────────────
# invoke()가 자체 생성한 체크리스트 문항 내용을 골드 체크리스트 문항과 LLM judge로 비교

gold_cl_map   = {row['set_id']: parse_json_cell(row['checklist'])['checklist']
                 for _, row in goldset_df.iterrows()}
v2_cl_map     = {sid: v['report'].get('checklist', []) for sid, v in v2_invoke_map.items()}

v2_cl_rows, v2_cl_errors = [], []
for _, row in goldset_df.iterrows():
    sid = row['set_id']
    if sid not in v2_cl_map:
        continue
    try:
        print(f'[Ver2 체크리스트 내용] set_id={sid}')
        v2_cl_rows.append(judge_checklist_content(sid, gold_cl_map[sid], v2_cl_map[sid]))
    except Exception as e:
        v2_cl_errors.append({'set_id': sid, 'error': repr(e)})
    time.sleep(0.2)

v2_cl_df = pd.DataFrame(v2_cl_rows)

print('\n─── Ver2 체크리스트 내용 지표 ───')
display(pd.DataFrame([{
    'count':          len(v2_cl_df),
    'pass_rate':      (v2_cl_df['pass_fail'] == 'PASS').mean(),
    'avg_overall':    v2_cl_df['overall_score'].mean(),
    'avg_matched':    v2_cl_df['matched_gold_count'].mean(),
    'avg_coverage':   v2_cl_df['coverage_score'].mean(),
    'avg_noise':      v2_cl_df['extra_noise_score'].mean(),
}]))
print()
show_checklist_content_ex(v2_cl_df, gold_cl_map, v2_cl_map)

v2_cl_df.to_csv(EVAL_DIR / 'eval_v2_checklist_content.csv', index=False, encoding='utf-8-sig')
if v2_cl_errors:
    display(pd.DataFrame(v2_cl_errors))


[Ver2 체크리스트 내용] set_id=0
[Ver2 체크리스트 내용] set_id=1
[Ver2 체크리스트 내용] set_id=2
[Ver2 체크리스트 내용] set_id=3
[Ver2 체크리스트 내용] set_id=4
[Ver2 체크리스트 내용] set_id=5
[Ver2 체크리스트 내용] set_id=6
[Ver2 체크리스트 내용] set_id=7
[Ver2 체크리스트 내용] set_id=8
[Ver2 체크리스트 내용] set_id=9
[Ver2 체크리스트 내용] set_id=10
[Ver2 체크리스트 내용] set_id=11
[Ver2 체크리스트 내용] set_id=12
[Ver2 체크리스트 내용] set_id=13
[Ver2 체크리스트 내용] set_id=14
[Ver2 체크리스트 내용] set_id=15
[Ver2 체크리스트 내용] set_id=16
[Ver2 체크리스트 내용] set_id=17
[Ver2 체크리스트 내용] set_id=18
[Ver2 체크리스트 내용] set_id=19

─── Ver2 체크리스트 내용 지표 ───


,count,pass_rate,avg_overall,avg_matched,avg_coverage,avg_noise
0,20,1.0,5.0,9.95,4.95,4.85



【 ▼ 내용 차이 큰 예시 】  set_id=0  overall=5  matched=10/10
  judge: 모든 항목이 골드 체크리스트와 잘 매칭되며, 평가의도 또한 동일하게 재현되었습니다. 일부 표현이 다를 수 있으나, 전반적으로 요구된 사항들을 충족했습니다.
   1. [골드] 학력 무관 조건 또는 이에 준하는 실무 역량을 갖추었는가?
      [생성] 학력에 제한이 없으나, 컴퓨터공학 전공 또는 관련 경험을 보유하고 있는가?
   2. [골드] 신입 가능 수준의 관련 개발 경험을 보유했는가?
      [생성] 신입도 지원 가능하지만, 관련 경력을 갖추었다면 더욱 유리한가?
   3. [골드] 필수 기술(Java, Spring Boot, MySQL, Kafka) 중 핵심 기술을 실제 프
      [생성] 필수 기술(Java, Spring Boot, MySQL, Kafka) 중 핵심 기술을 실제 프
   4. [골드] 우대 기술(Kubernetes, Redis, AWS) 경험 또는 빠른 학습 근거가 있는가?
      [생성] 우대 기술(Kubernetes, Redis, AWS) 중 하나 이상의 기술을 활용한 경험 또는
   5. [골드] 핀테크 도메인 또는 유사 서비스 업무 흐름을 이해하고 있는가?
      [생성] 루멘페이가 제공하는 결제 및 정산 플랫폼의 업무 흐름을 이해하고 있는가?
   6. [골드] REST API, 데이터 모델링, 배포/운영 등 서비스 개발 전반의 이해가 있는가?
      [생성] REST API, 데이터 모델링, 배포/운영 등 서비스 개발 전반에 대한 이해도가 있는가?
   7. [골드] 장애 대응이나 품질 개선 경험을 구체적으로 설명할 수 있는가?
      [생성] 장애 대응이나 품질 개선에 대한 어떤 경험이 있는가?
   8. [골드] 디자이너, PM, 운영 담당자 등 비개발 직군과 협업한 경험이 있는가?
      [생성] 디자이너, PM, 운영 담당자 등 비개발 직군과의 협업 경험이 있는가?
   9. [

---
## 2. 질문/모범답안 비교

| | Ver1 | Ver2 |
|--|------|------|
| 체크리스트 컨텍스트 | 골드 체크리스트(문항+result) 고정 | invoke() 자체 생성 |
| 질문 생성 | `make_interview_questions` | invoke() 내부 |

> 각 버전별로 LLM judge 점수 최저/최고 set의 첫 번째 질문/답안 쌍을 나란히 출력

In [22]:
# [Ver1-②] 고정 컨텍스트로 생성한 질문/모범답안을 골드와 judge 비교 → matched/coverage + 질문 예시.
# ── Ver1 질문 judge ────────────────────────────────────────────────────────

v1_q_rows, v1_q_errors = [], []
for _, row in goldset_df.iterrows():
    sid = row['set_id']
    if sid not in v1_questions_map:
        continue
    try:
        print(f'[Ver1 질문] set_id={sid}')
        v1_q_rows.append(judge_questions(sid, gold_qa_map[sid], v1_questions_map[sid]))
    except Exception as e:
        v1_q_errors.append({'set_id': sid, 'error': repr(e)})
    time.sleep(0.2)

v1_q_df = pd.DataFrame(v1_q_rows)
print('\n─── Ver1 질문 지표 ───')
display(q_metrics(v1_q_df))
print()
show_question_ex(v1_q_df, gold_qa_map, v1_questions_map)

v1_q_df.to_csv(EVAL_DIR / 'eval_v1_questions.csv', index=False, encoding='utf-8-sig')
if v1_q_errors:
    display(pd.DataFrame(v1_q_errors))

[Ver1 질문] set_id=0
[Ver1 질문] set_id=1
[Ver1 질문] set_id=2
[Ver1 질문] set_id=3
[Ver1 질문] set_id=4
[Ver1 질문] set_id=5
[Ver1 질문] set_id=6
[Ver1 질문] set_id=7
[Ver1 질문] set_id=8
[Ver1 질문] set_id=9
[Ver1 질문] set_id=10
[Ver1 질문] set_id=11
[Ver1 질문] set_id=12
[Ver1 질문] set_id=13
[Ver1 질문] set_id=14
[Ver1 질문] set_id=15
[Ver1 질문] set_id=16
[Ver1 질문] set_id=17
[Ver1 질문] set_id=18
[Ver1 질문] set_id=19

─── Ver1 질문 지표 ───


,count,pass_rate,avg_score,avg_matched,avg_coverage,avg_ans_sim
0,20,0.95,4.9,9.9,5.0,4.65



【 ▼ 낮은 유사도 예시 】  set_id=0  점수=4
  [골드 질문]  Java를 활용해 가장 복잡한 문제를 해결했던 경험을 설명해 주세요.
  [생성 질문]  필수 기술 중 Java를 활용해 복잡한 문제를 해결한 경험에 대해 말씀해 주세요.
  [골드 답안]  페이코어에서 간편결제 승인 API와 정산 배치 개발 과정의 병목을 분석하고 구조를 개선했습니다. 요구사항을 작은 단위로 나누고 테스트를 추가해 안정성을 높였습니다.
  [생성 답안]  페이코어에서 간편결제 승인 API를 개발하며 Java와 Spring Boot를 활용했습니다. 당시 결제 처리 과정에서 발생했던 비효율성을 분석하여 멱등성 키를 도입했습니다. 이를 

【 ▲ 높은 유사도 예시 】  set_id=1  점수=5
  [골드 질문]  TypeScript를 활용해 가장 복잡한 문제를 해결했던 경험을 설명해 주세요.
  [생성 질문]  JD 필수 기술 중 1순위 기술인 TypeScript를 활용해 복잡한 문제를 해결한 경험을 말씀해 주시겠습니까?
  [골드 답안]  웰니스앱에서 건강 기록 앱의 React Native 화면과 푸시 알림 개발 과정의 병목을 분석하고 구조를 개선했습니다. 요구사항을 작은 단위로 나누고 테스트를 추가해 안정성을 높였
  [생성 답안]  저는 웰니스앱에서 모바일 프론트엔드 개발자로 근무하면서 TypeScript를 사용해 동적 타입 체크를 통해 복잡한 건강 기록의 데이터를 안전하게 처리하는 작업을 수행했습니다. 특히



In [23]:
# [Ver2-②] invoke가 생성한 질문/모범답안을 골드와 judge 비교.
# ── Ver2 질문 judge ────────────────────────────────────────────────────────

v2_questions_map = {sid: v['questions'] for sid, v in v2_invoke_map.items()}

v2_q_rows, v2_q_errors = [], []
for _, row in goldset_df.iterrows():
    sid = row['set_id']
    if sid not in v2_questions_map:
        continue
    try:
        print(f'[Ver2 질문] set_id={sid}')
        v2_q_rows.append(judge_questions(sid, gold_qa_map[sid], v2_questions_map[sid]))
    except Exception as e:
        v2_q_errors.append({'set_id': sid, 'error': repr(e)})
    time.sleep(0.2)

v2_q_df = pd.DataFrame(v2_q_rows)
print('\n─── Ver2 질문 지표 ───')
display(q_metrics(v2_q_df))
print()
show_question_ex(v2_q_df, gold_qa_map, v2_questions_map)

v2_q_df.to_csv(EVAL_DIR / 'eval_v2_questions.csv', index=False, encoding='utf-8-sig')
if v2_q_errors:
    display(pd.DataFrame(v2_q_errors))

[Ver2 질문] set_id=0
[Ver2 질문] set_id=1
[Ver2 질문] set_id=2
[Ver2 질문] set_id=3
[Ver2 질문] set_id=4
[Ver2 질문] set_id=5
[Ver2 질문] set_id=6
[Ver2 질문] set_id=7
[Ver2 질문] set_id=8
[Ver2 질문] set_id=9
[Ver2 질문] set_id=10
[Ver2 질문] set_id=11
[Ver2 질문] set_id=12
[Ver2 질문] set_id=13
[Ver2 질문] set_id=14
[Ver2 질문] set_id=15
[Ver2 질문] set_id=16
[Ver2 질문] set_id=17
[Ver2 질문] set_id=18
[Ver2 질문] set_id=19

─── Ver2 질문 지표 ───


,count,pass_rate,avg_score,avg_matched,avg_coverage,avg_ans_sim
0,20,1.0,4.85,9.9,5.0,4.55



【 ▼ 낮은 유사도 예시 】  set_id=5  점수=4
  [골드 질문]  React를 활용해 가장 복잡한 문제를 해결했던 경험을 설명해 주세요.
  [생성 질문]  JD 필수 기술 중 1순위 기술인 React를 활용해 복잡한 문제를 해결한 경험에 대해 말씀해 주세요.
  [골드 답안]  게임포털에서 게임 이벤트 페이지와 계정 연동 UI 개발 과정의 병목을 분석하고 구조를 개선했습니다. 요구사항을 작은 단위로 나누고 테스트를 추가해 안정성을 높였습니다.
  [생성 답안]  지난 프로젝트에서 우리는 사용자 로그인 페이지의 성능 이슈를 겪었습니다. React를 활용하여 컴포넌트의 상태 관리를 최적화하고, 불필요한 렌더링을 줄여 성능을 개선했습니다. 이 

【 ▲ 높은 유사도 예시 】  set_id=0  점수=5
  [골드 질문]  Java를 활용해 가장 복잡한 문제를 해결했던 경험을 설명해 주세요.
  [생성 질문]  프로젝트에서 Java를 활용해 복잡한 문제를 해결했던 경험에 대해 이야기해 주세요.
  [골드 답안]  페이코어에서 간편결제 승인 API와 정산 배치 개발 과정의 병목을 분석하고 구조를 개선했습니다. 요구사항을 작은 단위로 나누고 테스트를 추가해 안정성을 높였습니다.
  [생성 답안]  저는 Java와 Spring Boot를 사용하여 복잡한 결제 시스템을 개발한 경험이 있습니다. 특히, 결제 장애를 줄이기 위해 멱등성 키와 재처리 큐를 도입했습니다. 이로 인해 오



In [24]:
# [비교표] Ver1 vs Ver2 질문 지표 나란히 — 차이가 크면 상류(요약/체크리스트 생성)가 병목.
# ── Ver1 vs Ver2 질문 비교표 ───────────────────────────────────────────────

print('─── Ver1 vs Ver2 질문 비교 ───')
display(pd.concat([
    q_metrics(v1_q_df).assign(version='Ver1'),
    q_metrics(v2_q_df).assign(version='Ver2'),
], ignore_index=True)[['version', 'count', 'pass_rate', 'avg_score', 'avg_matched', 'avg_coverage', 'avg_ans_sim']])

─── Ver1 vs Ver2 질문 비교 ───


,version,count,pass_rate,avg_score,avg_matched,avg_coverage,avg_ans_sim
0,Ver1,20,0.95,4.90,9.9,5.0,4.65
1,Ver2,20,1.00,4.85,9.9,5.0,4.55


---
## 3. 리포트 비교

| | Ver1 | Ver2 |
|--|------|------|
| 체크리스트 | 골드 체크리스트(문항+result) 고정 | invoke() 자체 생성 체크리스트 T/F |
| 리포트 생성 | `make_report` | invoke() 내부 |

> LLM judge가 grade_match, 내용 일치도, 환각 여부를 종합 평가  
> 점수 최저/최고 set의 등급·요약·최종의견을 골드 vs 생성으로 비교 출력

In [25]:
# [Ver1-③] 고정 컨텍스트로 생성한 리포트를 골드와 judge 비교 → grade_match/일치도 + 리포트 예시.
# ── Ver1 리포트 judge ──────────────────────────────────────────────────────

v1_r_rows, v1_r_errors = [], []
for _, row in goldset_df.iterrows():
    sid = row['set_id']
    if sid not in v1_reports_map:
        continue
    try:
        print(f'[Ver1 리포트] set_id={sid}')
        v1_r_rows.append(judge_report(sid, gold_rpt_map[sid], v1_reports_map[sid]))
    except Exception as e:
        v1_r_errors.append({'set_id': sid, 'error': repr(e)})
    time.sleep(0.2)

v1_r_df = pd.DataFrame(v1_r_rows)
print('\n─── Ver1 리포트 지표 ───')
display(r_metrics(v1_r_df))
print()
show_report_ex(v1_r_df, gold_rpt_map, v1_reports_map)

v1_r_df.to_csv(EVAL_DIR / 'eval_v1_report.csv', index=False, encoding='utf-8-sig')
if v1_r_errors:
    display(pd.DataFrame(v1_r_errors))

[Ver1 리포트] set_id=0
[Ver1 리포트] set_id=1
[Ver1 리포트] set_id=2
[Ver1 리포트] set_id=3
[Ver1 리포트] set_id=4
[Ver1 리포트] set_id=5
[Ver1 리포트] set_id=6
[Ver1 리포트] set_id=7
[Ver1 리포트] set_id=8
[Ver1 리포트] set_id=9
[Ver1 리포트] set_id=10
[Ver1 리포트] set_id=11
[Ver1 리포트] set_id=12
[Ver1 리포트] set_id=13
[Ver1 리포트] set_id=14
[Ver1 리포트] set_id=15
[Ver1 리포트] set_id=16
[Ver1 리포트] set_id=17
[Ver1 리포트] set_id=18
[Ver1 리포트] set_id=19

─── Ver1 리포트 지표 ───


,count,grade_match_rate,pass_rate,avg_score,avg_cl_match,avg_analysis_sim,avg_summary_sim
0,20,0.8,0.8,4.25,1.0,4.2,4.4



【 ▼ 낮은 유사도 예시 】  set_id=5  점수=3
  [골드] 등급=A
         요약: 오지훈 지원자는 React 중심의 개발 경험과 게임 직무와 연결 가능한 이력을 보유했습니다. 체크리스트 9/10개를 충족하여 A 수준으로 평가됩니다.
         의견: 오지훈 지원자는 게임 웹 서비스 프론트엔드 개발자 포지션에서 요구하는 역량 중 상당 부분을 충족합니다. 강점은 React 기반 문제 해결 경험이며, 부족 항목은 
  [생성] 등급=B
         요약: 체크리스트 9/10개를 충족하여 B 수준으로 평가됩니다.
         의견: 오지훈 님은 전반적으로 높은 기술력과 경험을 보유하고 있으며, 특정 분야에 대한 기본적인 이해도 추가 검증이 필요합니다.

【 ▲ 높은 유사도 예시 】  set_id=0  점수=5
  [골드] 등급=B
         요약: 김도현 지원자는 Java 중심의 개발 경험과 핀테크 직무와 연결 가능한 이력을 보유했습니다. 체크리스트 7/10개를 충족하여 B 수준으로 평가됩니다.
         의견: 김도현 지원자는 백엔드 결제 플랫폼 개발자 포지션에서 요구하는 역량 중 상당 부분을 충족합니다. 강점은 Java 기반 문제 해결 경험이며, 부족 항목은 면접에서 
  [생성] 등급=B
         요약: 체크리스트 7/10개를 충족하여 B 수준으로 평가됩니다.
         의견: 김도현 님은 전반적으로 높은 기술 역량을 가진 지원자이나 추가 검증이 필요한 부분이 있습니다.



In [26]:
# [Ver2-③] invoke가 생성한 리포트를 골드와 judge 비교.
# ── Ver2 리포트 judge ──────────────────────────────────────────────────────

v2_reports_map = {sid: v['report'] for sid, v in v2_invoke_map.items()}

v2_r_rows, v2_r_errors = [], []
for _, row in goldset_df.iterrows():
    sid = row['set_id']
    if sid not in v2_reports_map:
        continue
    try:
        print(f'[Ver2 리포트] set_id={sid}')
        v2_r_rows.append(judge_report(sid, gold_rpt_map[sid], v2_reports_map[sid]))
    except Exception as e:
        v2_r_errors.append({'set_id': sid, 'error': repr(e)})
    time.sleep(0.2)

v2_r_df = pd.DataFrame(v2_r_rows)
print('\n─── Ver2 리포트 지표 ───')
display(r_metrics(v2_r_df))
print()
show_report_ex(v2_r_df, gold_rpt_map, v2_reports_map)

v2_r_df.to_csv(EVAL_DIR / 'eval_v2_report.csv', index=False, encoding='utf-8-sig')
if v2_r_errors:
    display(pd.DataFrame(v2_r_errors))

[Ver2 리포트] set_id=0
[Ver2 리포트] set_id=1
[Ver2 리포트] set_id=2
[Ver2 리포트] set_id=3
[Ver2 리포트] set_id=4
[Ver2 리포트] set_id=5
[Ver2 리포트] set_id=6
[Ver2 리포트] set_id=7
[Ver2 리포트] set_id=8
[Ver2 리포트] set_id=9
[Ver2 리포트] set_id=10
[Ver2 리포트] set_id=11
[Ver2 리포트] set_id=12
[Ver2 리포트] set_id=13
[Ver2 리포트] set_id=14
[Ver2 리포트] set_id=15
[Ver2 리포트] set_id=16
[Ver2 리포트] set_id=17
[Ver2 리포트] set_id=18
[Ver2 리포트] set_id=19

─── Ver2 리포트 지표 ───


,count,grade_match_rate,pass_rate,avg_score,avg_cl_match,avg_analysis_sim,avg_summary_sim
0,20,0.4,0.4,3.3,0.805,3.6,3.55



【 ▼ 낮은 유사도 예시 】  set_id=6  점수=2
  [골드] 등급=A
         요약: 강유빈 지원자는 Go 중심의 개발 경험과 보안 직무와 연결 가능한 이력을 보유했습니다. 체크리스트 9/10개를 충족하여 A 수준으로 평가됩니다.
         의견: 강유빈 지원자는 클라우드 보안 플랫폼 엔지니어 포지션에서 요구하는 역량 중 상당 부분을 충족합니다. 강점은 Go 기반 문제 해결 경험이며, 부족 항목은 면접에서 
  [생성] 등급=B
         요약: 체크리스트 9/10개를 충족하여 B 수준으로 평가됩니다.
         의견: 지원자의 기술적 역량이 우수하나 서비스 개발 전반에 대한 추가 검증이 필요합니다.

【 ▲ 높은 유사도 예시 】  set_id=0  점수=5
  [골드] 등급=B
         요약: 김도현 지원자는 Java 중심의 개발 경험과 핀테크 직무와 연결 가능한 이력을 보유했습니다. 체크리스트 7/10개를 충족하여 B 수준으로 평가됩니다.
         의견: 김도현 지원자는 백엔드 결제 플랫폼 개발자 포지션에서 요구하는 역량 중 상당 부분을 충족합니다. 강점은 Java 기반 문제 해결 경험이며, 부족 항목은 면접에서 
  [생성] 등급=B
         요약: 체크리스트 7/10개를 충족하여 B 수준으로 평가됩니다.
         의견: 김도현 지원자는 기술적 역량과 문제 해결 능력에서 높은 가능성을 보여주지만, 일부 추가 경험이 필요한 분야가 있습니다.



In [27]:
# [비교표] Ver1 vs Ver2 리포트 지표 나란히.
# ── Ver1 vs Ver2 리포트 비교표 ─────────────────────────────────────────────

print('─── Ver1 vs Ver2 리포트 비교 ───')
display(pd.concat([
    r_metrics(v1_r_df).assign(version='Ver1'),
    r_metrics(v2_r_df).assign(version='Ver2'),
], ignore_index=True)[['version', 'count', 'grade_match_rate', 'pass_rate',
                        'avg_score', 'avg_cl_match', 'avg_analysis_sim', 'avg_summary_sim']])

─── Ver1 vs Ver2 리포트 비교 ───


,version,count,grade_match_rate,pass_rate,avg_score,avg_cl_match,avg_analysis_sim,avg_summary_sim
0,Ver1,20,0.8,0.8,4.25,1.000,4.2,4.40
1,Ver2,20,0.4,0.4,3.30,0.805,3.6,3.55


---
## 4. 종합 요약

In [28]:
# [종합 요약] 전 단계 핵심 지표를 한 표로. Ver1-Ver2 격차 = 상류 단 손실.
def _v(df, col):
    return round(df[col].iloc[0], 4) if len(df) and col in df.columns else float('nan')

v1_tf_m = tf_metrics(v1_tf_df)

# Ver2 체크리스트 내용 지표
v2_cl_m = pd.DataFrame([{
    'pass_rate':   (v2_cl_df['pass_fail'] == 'PASS').mean(),
    'avg_overall': v2_cl_df['overall_score'].mean(),
    'avg_matched': v2_cl_df['matched_gold_count'].mean(),
}])

summary = pd.DataFrame([
    {'버전': 'Ver2', '평가 단계': '⓪ 요약',           '지표': 'pass_rate',   '값': round((sum_judge_df['pass_fail'] == 'PASS').mean(), 4)},
    {'버전': 'Ver2', '평가 단계': '⓪ 요약',           '지표': 'avg_score',   '값': round(sum_judge_df['overall_score'].mean(), 4)},
    {'버전': 'Ver1', '평가 단계': '① 체크리스트 T/F',  '지표': 'accuracy',    '값': _v(v1_tf_m, 'accuracy')},
    {'버전': 'Ver1', '평가 단계': '① 체크리스트 T/F',  '지표': 'f1',          '값': _v(v1_tf_m, 'f1')},
    {'버전': 'Ver1', '평가 단계': '② 질문/모범답안',    '지표': 'pass_rate',   '값': _v(q_metrics(v1_q_df), 'pass_rate')},
    {'버전': 'Ver1', '평가 단계': '② 질문/모범답안',    '지표': 'avg_score',   '값': _v(q_metrics(v1_q_df), 'avg_score')},
    {'버전': 'Ver1', '평가 단계': '③ 리포트',           '지표': 'grade_match', '값': _v(r_metrics(v1_r_df), 'grade_match_rate')},
    {'버전': 'Ver1', '평가 단계': '③ 리포트',           '지표': 'pass_rate',   '값': _v(r_metrics(v1_r_df), 'pass_rate')},
    {'버전': 'Ver1', '평가 단계': '③ 리포트',           '지표': 'avg_score',   '값': _v(r_metrics(v1_r_df), 'avg_score')},
    {'버전': 'Ver2', '평가 단계': '① 체크리스트 내용',  '지표': 'pass_rate',   '값': _v(v2_cl_m, 'pass_rate')},
    {'버전': 'Ver2', '평가 단계': '① 체크리스트 내용',  '지표': 'avg_matched', '값': _v(v2_cl_m, 'avg_matched')},
    {'버전': 'Ver2', '평가 단계': '① 체크리스트 내용',  '지표': 'avg_score',   '값': _v(v2_cl_m, 'avg_overall')},
    {'버전': 'Ver2', '평가 단계': '② 질문/모범답안',    '지표': 'pass_rate',   '값': _v(q_metrics(v2_q_df), 'pass_rate')},
    {'버전': 'Ver2', '평가 단계': '② 질문/모범답안',    '지표': 'avg_score',   '값': _v(q_metrics(v2_q_df), 'avg_score')},
    {'버전': 'Ver2', '평가 단계': '③ 리포트',           '지표': 'grade_match', '값': _v(r_metrics(v2_r_df), 'grade_match_rate')},
    {'버전': 'Ver2', '평가 단계': '③ 리포트',           '지표': 'pass_rate',   '값': _v(r_metrics(v2_r_df), 'pass_rate')},
    {'버전': 'Ver2', '평가 단계': '③ 리포트',           '지표': 'avg_score',   '값': _v(r_metrics(v2_r_df), 'avg_score')},
])

display(summary.pivot_table(
    index=['버전', '평가 단계'], columns='지표', values='값', aggfunc='first'
).round(4))

summary.to_csv(EVAL_DIR / 'eval_v1v2_summary.csv', index=False, encoding='utf-8-sig')
print('\n저장 완료:')
print('  eval_v1_tf.csv, eval_v1_questions.csv, eval_v1_report.csv')
print('  eval_v2_summary_stage.csv, eval_v2_checklist_content.csv, eval_v2_questions.csv, eval_v2_report.csv')
print('  eval_v1v2_summary.csv')


지표                accuracy  avg_matched  avg_score     f1  grade_match  \
버전   평가 단계                                                               
Ver1 ① 체크리스트 T/F     0.725          NaN        NaN  0.811          NaN   
     ② 질문/모범답안         NaN          NaN       4.90    NaN          NaN   
     ③ 리포트             NaN          NaN       4.25    NaN          0.8   
Ver2 ① 체크리스트 내용        NaN         9.95       5.00    NaN          NaN   
     ② 질문/모범답안         NaN          NaN       4.85    NaN          NaN   
     ③ 리포트             NaN          NaN       3.30    NaN          0.4   
     ⓪ 요약              NaN          NaN       4.15    NaN          NaN   

지표                pass_rate  
버전   평가 단계                   
Ver1 ① 체크리스트 T/F        NaN  
     ② 질문/모범답안         0.95  
     ③ 리포트             0.80  
Ver2 ① 체크리스트 내용        1.00  
     ② 질문/모범답안         1.00  
     ③ 리포트             0.40  
     ⓪ 요약              0.35


저장 완료:
  eval_v1_tf.csv, eval_v1_questions.csv, eval_v1_report.csv
  eval_v2_summary_stage.csv, eval_v2_checklist_content.csv, eval_v2_questions.csv, eval_v2_report.csv
  eval_v1v2_summary.csv
